# Copper Supply Globe: Port-to-Port Flows

Interactive 3D globe of seaborne copper flows (concentrate and cathode), built with **Plotly**.
- Ports are the end points, routes are continuous great-circle lines
- Line width scales with volume (kt/yr)
- Hover any route to see the supply between the two ports
- Buttons filter by product type, and the globe can be dragged to rotate

> Volumes are illustrative estimates. Replace the data in section 2 (or load your own CSV) for real analysis.

In [ ]:
# Run once if needed
# %pip install plotly pandas numpy

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

## 1. Data

In [ ]:
# port: (lat, lon, country)
PORTS = {
    "Antofagasta": (-23.65, -70.40, "Chile"), "Mejillones": (-23.10, -70.45, "Chile"),
    "San Antonio": (-33.59, -71.61, "Chile"), "Callao": (-12.05, -77.15, "Peru"),
    "Matarani": (-17.00, -72.11, "Peru"), "Vancouver": (49.29, -123.11, "Canada"),
    "Port Adelaide": (-34.80, 138.50, "Australia"), "Amamapare": (-4.83, 136.83, "Indonesia"),
    "Beira": (-19.83, 34.84, "Mozambique"), "Dar es Salaam": (-6.83, 39.29, "Tanzania"),
    "Durban": (-29.87, 31.03, "South Africa"), "Tianjin": (38.99, 117.75, "China"),
    "Qingdao": (36.07, 120.32, "China"), "Shanghai": (30.63, 122.06, "China"),
    "Ningbo": (29.87, 121.55, "China"), "Nansha": (22.75, 113.60, "China"),
    "Chiba": (35.60, 140.10, "Japan"), "Onsan": (35.47, 129.37, "South Korea"),
    "Mundra": (22.74, 69.70, "India"), "Hamburg": (53.54, 9.98, "Germany"),
    "Huelva": (37.26, -6.95, "Spain"), "Rotterdam": (51.95, 4.14, "Netherlands"),
    "Los Angeles": (33.73, -118.26, "USA"),
}

# origin, destination, volume (kt/yr), product type
ROUTES = pd.DataFrame([
    ("Antofagasta","Shanghai",520,"Cathode"), ("Mejillones","Qingdao",430,"Concentrate"),
    ("Antofagasta","Tianjin",360,"Concentrate"), ("San Antonio","Ningbo",310,"Cathode"),
    ("Antofagasta","Chiba",280,"Concentrate"), ("Mejillones","Onsan",240,"Concentrate"),
    ("Antofagasta","Hamburg",150,"Cathode"), ("San Antonio","Rotterdam",120,"Cathode"),
    ("Antofagasta","Mundra",170,"Concentrate"), ("Callao","Tianjin",340,"Concentrate"),
    ("Matarani","Nansha",260,"Concentrate"), ("Callao","Chiba",190,"Concentrate"),
    ("Matarani","Huelva",140,"Concentrate"), ("Callao","Hamburg",90,"Concentrate"),
    ("Amamapare","Chiba",230,"Concentrate"), ("Amamapare","Onsan",180,"Concentrate"),
    ("Amamapare","Mundra",120,"Concentrate"), ("Port Adelaide","Shanghai",160,"Cathode"),
    ("Port Adelaide","Onsan",110,"Cathode"), ("Dar es Salaam","Shanghai",280,"Cathode"),
    ("Beira","Nansha",180,"Cathode"), ("Durban","Tianjin",130,"Cathode"),
    ("Vancouver","Qingdao",200,"Concentrate"), ("Vancouver","Chiba",100,"Concentrate"),
    ("Los Angeles","Ningbo",95,"Cathode"),
], columns=["origin", "destination", "volume_kt", "type"])

# To use your own data instead:
# ROUTES = pd.read_csv("routes.csv")   # columns: origin, destination, volume_kt, type

## 2. Summary

In [ ]:
total = ROUTES.volume_kt.sum()
print(f"Total volume: {total:,} kt/yr | Routes: {len(ROUTES)} | Ports: {len(set(ROUTES.origin) | set(ROUTES.destination))}")

display(ROUTES.groupby("type").volume_kt.sum().to_frame("kt/yr"))
display(ROUTES.groupby("origin").volume_kt.sum().sort_values(ascending=False).head(5).to_frame("Top exporters (kt/yr)"))
display(ROUTES.groupby("destination").volume_kt.sum().sort_values(ascending=False).head(5).to_frame("Top importers (kt/yr)"))

## 3. Build the globe

In [ ]:
COLORS = {"Concentrate": "#e8943a", "Cathode": "#4fd1c5"}
PORT_ORIGIN, PORT_DEST = "#ffb866", "#7ff5ea"

def great_circle(lat1, lon1, lat2, lon2, n=60):
    """Points along the great-circle path between two coordinates."""
    p1 = np.radians([lat1, lon1]); p2 = np.radians([lat2, lon2])
    v1 = np.array([np.cos(p1[0])*np.cos(p1[1]), np.cos(p1[0])*np.sin(p1[1]), np.sin(p1[0])])
    v2 = np.array([np.cos(p2[0])*np.cos(p2[1]), np.cos(p2[0])*np.sin(p2[1]), np.sin(p2[0])])
    omega = np.arccos(np.clip(v1 @ v2, -1, 1))
    t = np.linspace(0, 1, n)[:, None]
    v = (np.sin((1-t)*omega)*v1 + np.sin(t*omega)*v2) / np.sin(omega)
    return np.degrees(np.arcsin(v[:, 2])), np.degrees(np.arctan2(v[:, 1], v[:, 0]))

lane_total = ROUTES.groupby(["origin", "destination"]).volume_kt.sum()
fig = go.Figure()
trace_types = []          # product type of every trace, used by the filter buttons

for r in ROUTES.sort_values("volume_kt").itertuples():
    lat1, lon1, c1 = PORTS[r.origin]; lat2, lon2, c2 = PORTS[r.destination]
    lats, lons = great_circle(lat1, lon1, lat2, lon2)
    share = lane_total[(r.origin, r.destination)] / total * 100
    hover = (f"<b>{r.origin} → {r.destination}</b><br>{c1} → {c2}<br>"
             f"Supply: <b>{r.volume_kt:,} kt/yr</b> ({r.type})<br>Share of network: {share:.1f}%")
    # continuous route line
    fig.add_trace(go.Scattergeo(lat=lats, lon=lons, mode="lines", hoverinfo="skip",
        line=dict(width=1 + r.volume_kt/150, color=COLORS[r.type]), opacity=0.85, showlegend=False))
    trace_types.append(r.type)
    # near-invisible markers along the route so hovering anywhere on the line shows the supply
    fig.add_trace(go.Scattergeo(lat=lats, lon=lons, mode="markers", hovertemplate=hover + "<extra></extra>",
        marker=dict(size=9, color="rgba(255,255,255,0.01)"), showlegend=False))
    trace_types.append(r.type)

# ports
exp = ROUTES.groupby("origin").volume_kt.sum(); imp = ROUTES.groupby("destination").volume_kt.sum()
ports = pd.DataFrame([(n, *PORTS[n], exp.get(n, 0), imp.get(n, 0)) for n in set(exp.index) | set(imp.index)],
                     columns=["port", "lat", "lon", "country", "exports", "imports"])
ports["is_origin"] = ports.exports > 0
ports["text"] = ports.apply(lambda p: f"<b>{p.port}</b>, {p.country}<br>" +
    (f"Exports: {p.exports:,} kt/yr" if p.is_origin else f"Imports: {p.imports:,} kt/yr"), axis=1)
fig.add_trace(go.Scattergeo(lat=ports.lat, lon=ports.lon, mode="markers+text", text=ports.port,
    textposition="top center", textfont=dict(size=9, color="#c9d4e6"), hovertext=ports.text, hoverinfo="text",
    marker=dict(size=7, color=[PORT_ORIGIN if o else PORT_DEST for o in ports.is_origin],
                line=dict(width=1, color="white")), name="Ports", showlegend=False))
trace_types.append("Ports")

def visibility(kind):
    return [(t == "Ports") or kind == "All" or t == kind for t in trace_types]

fig.update_layout(
    title=dict(text="<b>Cu</b> Copper Supply Globe: port-to-port flows (kt/yr)", x=0.5, font=dict(color="#e6edf7")),
    paper_bgcolor="#070b14", height=760, margin=dict(l=0, r=0, t=60, b=0),
    hoverlabel=dict(bgcolor="#0b1220", font_color="#e6edf7", bordercolor="#22304a"),
    geo=dict(projection_type="orthographic", projection_rotation=dict(lon=-20, lat=15),
             showland=True, landcolor="#16213a", showocean=True, oceancolor="#0a1426",
             showcountries=True, countrycolor="#2a3a5a", showcoastlines=True, coastlinecolor="#3a4d75",
             showlakes=False, bgcolor="#070b14", lataxis_showgrid=True, lonaxis_showgrid=True,
             lataxis_gridcolor="#142038", lonaxis_gridcolor="#142038"),
    updatemenus=[dict(type="buttons", direction="right", x=0.5, xanchor="center", y=0.02,
        bgcolor="#0b1220", font=dict(color="#e6edf7"), bordercolor="#22304a",
        buttons=[dict(label=k, method="update", args=[{"visible": visibility(k)}]) for k in ["All", "Concentrate", "Cathode"]])],
    annotations=[dict(text="<span style='color:#e8943a'>■</span> Concentrate &nbsp; <span style='color:#4fd1c5'>■</span> Cathode &nbsp; "
                           "|&nbsp; drag to rotate, scroll to zoom, hover a route for supply",
                      x=0.5, y=0.99, xref="paper", yref="paper", showarrow=False, font=dict(size=12, color="#8a9bb8"))],
)
fig.show()

## 4. Export for GitHub Pages
Writes a standalone HTML file. Commit it as `docs/index.html` (or `index.html`) and enable **Settings → Pages** to get a public URL.

In [ ]:
import os
os.makedirs("docs", exist_ok=True)
fig.write_html("docs/index.html", include_plotlyjs="cdn", full_html=True)
print("Saved docs/index.html")

## 5. Publish
```bash
git init && git add . && git commit -m "Copper supply globe"
git branch -M main
git remote add origin https://github.com/<your-username>/copper-supply-globe.git
git push -u origin main
```
Then **Settings → Pages → Deploy from a branch → `main` / `docs`**. Your dashboard will be live at `https://<your-username>.github.io/copper-supply-globe/`.